In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Tạo tensor ngẫu nhiên đại diện cho ảnh 1 kênh 4x4
x = torch.rand(4, 4)
print("Tensor x:\n", x)
print(f"Shape: {x.shape}, Dtype: {x.dtype}, Device: {x.device}")

Tensor x:
 tensor([[0.7382, 0.6714, 0.6148, 0.7249],
        [0.6327, 0.3018, 0.8003, 0.7380],
        [0.7711, 0.8991, 0.2088, 0.9394],
        [0.3370, 0.9217, 0.5965, 0.6951]])
Shape: torch.Size([4, 4]), Dtype: torch.float32, Device: cpu


In [5]:
# Giả lập 1 ảnh RGB kích thước 100x100 từ NumPy (H, W, C)
dummy_img_np = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)
print("1. Shape ảnh NumPy ban đầu (H, W, C):", dummy_img_np.shape)

# Chuyển NumPy -> PyTorch Tensor & chuẩn hóa về [0.0, 1.0]
tensor_img = torch.from_numpy(dummy_img_np).float() / 255.0

# Hoán đổi chiều [H, W, C] -> [C, H, W] để đúng chuẩn PyTorch
tensor_img = tensor_img.permute(2, 0, 1)
print("2. Shape sau permute (C, H, W):", tensor_img.shape)

# Thêm chiều Batch [C, H, W] -> [B, C, H, W]
tensor_batch = tensor_img.unsqueeze(0)
print("3. Shape sau khi thêm Batch size (B, C, H, W):", tensor_batch.shape)

# Loại bỏ chiều Batch để vẽ lại bằng Matplotlib
tensor_back = tensor_batch.squeeze(0).permute(1, 2, 0)
print("4. Shape trả về hiển thị (H, W, C):", tensor_back.shape)

1. Shape ảnh NumPy ban đầu (H, W, C): (100, 100, 3)
2. Shape sau permute (C, H, W): torch.Size([3, 100, 100])
3. Shape sau khi thêm Batch size (B, C, H, W): torch.Size([1, 3, 100, 100])
4. Shape trả về hiển thị (H, W, C): torch.Size([100, 100, 3])


In [6]:
# Giả sử ta có dữ liệu x = 2.0, nhãn thực tế y = 10.0
# Mô hình: y_pred = w * x + b
w = torch.tensor([1.5], requires_grad=True)
b = torch.tensor([0.5], requires_grad=True)
x = torch.tensor([2.0])
y_true = torch.tensor([10.0])

# 1. Forward Pass
y_pred = w * x + b

# 2. Hàm mất mát (MSE Loss: (y_pred - y_true)^2)
loss = (y_pred - y_true) ** 2
print(f"Loss ban đầu: {loss.item():.4f}")

loss.backward()

# 4. In đạo hàm dLoss/dw và dLoss/db
print(f"dL/dw: {w.grad.item():.4f}")  # 2 * (y_pred - y_true) * x = 2 * (3.5 - 10) * 2 = -26.0
print(f"dL/db: {b.grad.item():.4f}")  # 2 * (y_pred - y_true) * 1 = 2 * (3.5 - 10) * 1 = -13.0

Loss ban đầu: 42.2500
dL/dw: -26.0000
dL/db: -13.0000


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Giả lập dữ liệu: 4 mẫu (Batch size = 4), mỗi mẫu có 5 đặc trưng
X = torch.randn(4, 5)
# Nhãn thực tế tương ứng với 3 lớp (0, 1, 2)
y_true = torch.tensor([0, 2, 1, 0], dtype=torch.long)

# 2. Xây dựng mô hình tuyến tính đơn giản: 5 features -> 3 logits
model = nn.Linear(in_features=5, out_features=3)

# 3. Định nghĩa Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-4)

# 4. Huấn luyện 5 bước (Iterations)
for epoch in range(1, 10):
    # Bước 1: Xóa sạch gradient cũ tích lũy từ bước trước
    optimizer.zero_grad()
    
    # Bước 2: Forward pass (trả về raw logits [4, 3])
    logits = model(X)
    
    # Bước 3: Tính Loss trực tiếp từ Logits và Ground Truth
    loss = criterion(logits, y_true)
    
    # Bước 4: Backward pass (tính đạo hàm dL/dW)
    loss.backward()
    
    # Bước 5: Cập nhật trọng số theo AdamW
    optimizer.step()
    
    # Lấy nhãn dự đoán (lớp có logit lớn nhất)
    with torch.no_grad():
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y_true).float().mean().item()
        
    print(f"Epoch {epoch}: Loss = {loss.item():.4f} | Accuracy = {acc * 100:.1f}%")

Epoch 1: Loss = 1.2670 | Accuracy = 25.0%
Epoch 2: Loss = 1.2264 | Accuracy = 25.0%
Epoch 3: Loss = 1.1866 | Accuracy = 25.0%
Epoch 4: Loss = 1.1477 | Accuracy = 25.0%
Epoch 5: Loss = 1.1098 | Accuracy = 25.0%
Epoch 6: Loss = 1.0728 | Accuracy = 25.0%
Epoch 7: Loss = 1.0367 | Accuracy = 25.0%
Epoch 8: Loss = 1.0015 | Accuracy = 50.0%
Epoch 9: Loss = 0.9672 | Accuracy = 50.0%


Các tham số quan trọng:
- Kernel ($K$): Ma trận trọng số học được (thường là $3 \times 3$ hoặc $5 \times 5$).
- Stride ($S$): Bước nhảy của Kernel sau mỗi phép tính tích chập. $S=1$ quét từng pixel; $S=2$ giảm một nửa kích thước ảnh.
- Padding ($P$): Thêm viền giá trị $0$ quanh ảnh để giữ nguyên kích thước không gian và tránh mất thông tin ở các mép ngoài.

Công thức tính kích thước đầu ra:
$$O = \left\lfloor \frac{I - K + 2P}{S} \right\rfloor + 1$$

- $I$: Kích thước cạnh đầu vào ($H_{\text{in}}$ hoặc $W_{\text{in}}$)
- $O$: Kích thước cạnh đầu ra ($H_{\text{out}}$ hoặc $W_{\text{out}}$)


In [30]:
import torch
import torch.nn as nn

# Giả lập 1 Batch gồm 2 ảnh màu RGB kích thước 32x32: [B, C, H, W]
x = torch.randn(2, 3, 32, 32)
print("0. Kích thước đầu vào ban đầu:", x.shape)

# Tầng 1: Conv2d (In=3, Out=16, Kernel=3, Stride=1, Padding=1)
# O = floor((32 - 3 + 2*1) / 1) + 1 = 32
conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
act1 = nn.ReLU()
x1 = act1(conv1(x))
print("1. Sau Conv1 (K=3, S=1, P=1) + ReLU:", x1.shape)

# Tầng 2: MaxPool2d (Kernel=2, Stride=2)
# O = floor((32 - 2 + 0) / 2) + 1 = 16
pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
x2 = pool1(x1)
print("2. Sau MaxPool (K=2, S=2):", x2.shape)

# Tầng 3: Conv2d (In=16, Out=32, Kernel=5, Stride=1, Padding=0)
# O = floor((16 - 5 + 0) / 1) + 1 = 12
conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=0)
act2 = nn.ReLU()
x3 = act2(conv2(x2))
print("3. Sau Conv2 (K=5, S=1, P=0) + ReLU:", x3.shape)

# Tầng 4: Flatten để chuyển sang Vector 1D cho Fully Connected Layer
flatten = nn.Flatten()
x4 = flatten(x3)
print("4. Sau Flatten để nối vào Linear layer:", x4.shape)
# Kích thước vector: 32 channels * 12 height * 12 width = 4608 features

0. Kích thước đầu vào ban đầu: torch.Size([2, 3, 32, 32])
1. Sau Conv1 (K=3, S=1, P=1) + ReLU: torch.Size([2, 16, 32, 32])
2. Sau MaxPool (K=2, S=2): torch.Size([2, 16, 16, 16])
3. Sau Conv2 (K=5, S=1, P=0) + ReLU: torch.Size([2, 32, 12, 12])
4. Sau Flatten để nối vào Linear layer: torch.Size([2, 4608])
